# Yahoo Finance ('yfinance') Web Scrapping Notebook
This notebook will be the introduction to working with the yfinance api for webscrapping. We will predominantly be using this for finding ETF price and volume data but we will need to adjust it for divdends. The best thing I think we can do is divide into sectors, but also pull full index (SPY, QQQ), we will then need to find proxies for different maturity bonds (long and short) and the equivalent of a money market (1-3 month treasuries). It may be a good idea to pull currency data from this as well, possibly the DXY index of USD strength.

## Libraries

In [14]:
import numpy as np
import pandas as pd
import altair as alt  

import yfinance as yf

# Disable the max rows limit in Altair
alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

ALright, let's start with just the basic pulls. I am pretty sure we should be able to pull the data using a batch pull but if not, let's pull them indvidually and add them to dataframes via a horizontal merge on date. We will only be focusing on closing prices and volume traded in the day. We can decide any data tansformations we want to use in the future.

In [15]:
# Let's create a list of sectors
sector_etfs = [
    # Broad equity
    "SPY", "QQQ", "IWM",
    # GICS sectors
    "XLF", "XLK", "XLU", "XLV", "XLE", "XLI", "XLB", "XLP", "XLY",
    # Fixed income — short / intermediate / long / credit / inflation
    "BIL", "BND", "IEF", "TLT", "LQD", "HYG", "TIP",
    # Real assets
    "XLRE", "GLD"
]

prices = yf.download(sector_etfs, period='max', auto_adjust=True)['Close']

[*********************100%***********************]  21 of 21 completed


I'm going to separate the data pull from the data analysis so I don't have to continue to query the API as rate limiting is a known issue.

In [16]:
prices.dropna(inplace=True)
prices.head()



Ticker,BIL,BND,GLD,HYG,IEF,IWM,LQD,QQQ,SPY,TIP,...,XLB,XLE,XLF,XLI,XLK,XLP,XLRE,XLU,XLV,XLY
Date,,,,,,,,,,,,,,,,,,,,,
2015-10-08,74.501579,60.672009,109.139999,49.211655,85.558228,101.028694,81.140396,98.336395,169.524307,83.937294,...,17.947430,22.906578,15.673987,44.282928,18.358355,37.508663,21.185570,15.831950,57.437981,34.710793
2015-10-09,74.501579,60.664577,110.870003,49.217461,85.606033,101.229858,81.175339,98.781464,169.625366,83.704117,...,17.939301,22.758038,15.573471,44.432758,18.438370,37.592262,21.150505,15.756386,57.699341,34.751038
2015-10-12,74.501579,60.768463,111.309998,49.124916,85.892906,101.081184,81.503777,99.022545,169.785477,83.854561,...,17.784895,22.464230,15.586871,44.441071,18.460588,37.691071,21.283749,15.896710,57.851082,34.916473
2015-10-13,74.501579,60.805523,111.860001,49.009247,85.972595,99.699272,81.440903,98.382759,168.715454,83.809425,...,17.715816,22.229849,15.466253,43.966606,18.420582,37.463066,21.150505,15.860723,57.126068,34.728680
2015-10-14,74.501579,61.028069,113.809998,48.974548,86.442665,98.719673,81.762360,98.225136,167.906616,84.095276,...,17.858040,22.421310,15.338930,43.492146,18.385019,37.029823,21.150505,15.857131,57.016457,34.375439


In [17]:
prices.tail()

Ticker,BIL,BND,GLD,HYG,IEF,IWM,LQD,QQQ,SPY,TIP,...,XLB,XLE,XLF,XLI,XLK,XLP,XLRE,XLU,XLV,XLY
Date,,,,,,,,,,,,,,,,,,,,,
2026-03-02,91.389999,74.639999,490.000000,80.279999,97.120003,263.809998,110.919998,608.090027,686.380005,111.570000,...,53.250000,57.040001,51.299999,178.899994,139.539993,88.709999,43.919998,47.369999,158.529999,115.419998
2026-03-03,91.400002,74.559998,468.140015,80.120003,97.010002,259.239990,110.870003,601.580017,680.330017,111.500000,...,51.939999,56.520000,51.209999,175.440002,137.500000,87.739998,43.700001,47.070000,156.740005,114.360001
2026-03-04,91.410004,74.510002,471.799988,80.400002,96.809998,261.760010,110.970001,610.750000,685.130005,111.250000,...,51.919998,56.189999,51.500000,175.970001,139.839996,87.160004,43.759998,47.270000,157.050003,116.389999
2026-03-05,91.419998,74.339996,466.130005,80.080002,96.510002,256.760010,110.529999,608.909973,681.309998,111.220001,...,50.830002,56.480000,51.230000,172.059998,140.179993,85.410004,43.340000,46.900002,153.910004,116.550003
2026-03-06,91.441101,74.355003,474.773193,79.940002,96.660004,251.690002,110.410103,603.320007,674.700012,111.620102,...,49.980000,56.709999,50.375000,170.089996,138.725006,85.529999,42.939999,46.900002,152.449997,114.720001


Alright, That data pull should work pretty easily let's take a look at much historical data we have. I thinmk the biggest concern is the fact that if we drop NA's we only have sector data from 2018. This does not give us a lot of exposure do different market regimes. I image this is do to the relatively new explosion of ETFs, and I imagine fixed income ETF's may contribute more to this problem. We may want to consider other options as surrogates.

In [18]:
min_date = min(prices.index)
print(f"The earliest date in our data is {min_date}.")

The earliest date in our data is 2015-10-08 00:00:00.


Ryan Peet brought up a really good idea of using mutual funds as proxies as they have been investment vehicles for a much longer period of time. So let's see if we can build the same datapull for mutual funds, and see how far back that data goes.  
Broad Market (SPY proxy): Vanguard 500 Index (VFINX) — data back to 1976, the gold standard  
Tech (XLK): Fidelity Select Technology (FSPTX) — inception 1981  
Healthcare (XLV): Fidelity Select Health Care (FSPHX) — inception 1981  
Energy (XLE): Fidelity Select Energy (FSENX) — inception 1981  
Financials (XLF): Fidelity Select Financial Services (FIDSX) — inception 1981  
Utilities (XLU): Fidelity Select Utilities (FSUTX) — inception 1981  
*Note* Industrials is really hard because it wasn't really a sector until the late 90's early '00s. It may be best to just drop it as the only one that works is a very heavily weighted subsection of industrials  
Industrials (XLI): Fidelity Select Industrials (FCYIX) — inception 1997 (this one is shorter I actually could only get data to 2019)
Industrials2 (XLI): Fidelity Select Defense & Aerospace (FSDAX)  
Consumer Staples (XLP): Fidelity Select Consumer Staples (FDFAX) — inception 1985  
Consumer Discretionary (XLY): Fidelity Select Retailing (FSRPX) as an imperfect proxy  
Materials (XLB): Fidelity Select Materials (FSDPX) — inception 1986  
Bonds (short-term): Vanguard Short-Term Bond Index (VBISX) or use direct Treasury yields from FRED  
Bonds (long-term): Vanguard Long-Term Bond Index (VBLTX) or TLT equivalent via Barclays index data from FRED  
Money market: 3-month T-bill rate from FRED is cleaner than any fund proxy  


In [19]:
# Let's create a list of sectors
sector_mfs = ["VFINX","FSPTX","FSPHX","FSENX","FIDSX","FSUTX","FSDAX","FDFAX","FSRPX","FSDPX","VBISX","VBLTX"]
# I am going to comment out the mutual fund pull because we decided against them
#prices_mfs = yf.download(sector_mfs, period='max', auto_adjust=True)[['Close','Volume']]

In [20]:
#prices_mfs.dropna(inplace=True)
#prices_mfs.head()



In [21]:
#prices_mfs.tail()

Alright, based on this analysis, I still think that the ETF's are the strongest approach. Let's also look at some different indices that may be beneficial to our understanding of the current environment (independent variables).  

Volatility & Fear  
^VIX — CBOE Volatility Index (equity fear gauge)  
^VXN — Nasdaq volatility equivalent  
^MOVE — Bond market volatility (the "VIX for Treasuries")  

Currency  

DX-Y.NYB — DXY Dollar Index  
EURUSD=X, JPYUSD=X, CNYUSD=X — Major pairs (EUR, Yen, Yuan signal global risk appetite and trade conditions)  

Rates & Credit  

^TNX — 10-Year Treasury yield  
^TYX — 30-Year Treasury yield  
^IRX — 13-week T-Bill (short end)  
^FVX — 5-Year Treasury yield  

Commodities (macro signals)

GC=F — Gold (inflation hedge / flight to safety)  
CL=F — Crude Oil WTI (growth proxy, geopolitical risk)  
NG=F — Natural Gas  
HG=F — Copper ("Dr. Copper" — leading economic indicator)  

Credit Spreads (via ETFs since yfinance doesn't have spread data directly)  

HYG — High Yield Corporate Bonds (risk appetite)  
LQD — Investment Grade Corporate Bonds  
TLT — Long Duration Treasuries (rate sensitivity)  
SHY — Short Duration Treasuries  

Global / Geopolitical

^FTSE — UK (Brexit/European stability)  
^N225 — Nikkei (Japan / Asia Pacific) #Not working so lets try futures
NKY=F - Nikkei Futures (not spot price) Also not working so lets remove   
^HSI — Hang Seng (China exposure)  
^GSPC — S&P 500 broad market  

In [22]:
indicies = ["^VIX","^VXN","^MOVE","DX-Y.NYB","^TNX","GC=F","CL=F","NG=F","HG=F","HYG","LQD","TLT","SHY","^FTSE","^HSI","^GSPC"] # When we removed the Credit spreads we went back to 2002, If we remove the VIX and VXN and MOVE then we can go back to late 2000.

prices_indicies = yf.download(indicies, period='max', auto_adjust=True)['Close']
# We may need to run the data with more historical data but less features and more features but less historical data to see what works best for our model but keep in mind the impact that it will have on out of sample data.

[*********************100%***********************]  16 of 16 completed


In [23]:
#prices_indicies.dropna(inplace=True)
prices_indicies.head()

Ticker,CL=F,DX-Y.NYB,GC=F,HG=F,HYG,LQD,NG=F,SHY,TLT,^FTSE,^GSPC,^HSI,^MOVE,^TNX,^VIX,^VXN
Date,,,,,,,,,,,,,,,,
1927-12-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,17.660000,NaN,NaN,NaN,NaN,NaN
1928-01-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,17.760000,NaN,NaN,NaN,NaN,NaN
1928-01-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,17.719999,NaN,NaN,NaN,NaN,NaN
1928-01-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,17.549999,NaN,NaN,NaN,NaN,NaN
1928-01-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,17.660000,NaN,NaN,NaN,NaN,NaN


This amount of data only goes back to 2007, so we may want to consider dropping some of them to see if we can get data back to our origination of the macro and ETF data (around 1994)

Let's merge the data that we have here into a single dataframe and export it as a csv. 

In [24]:
print(prices.columns)
print(prices_indicies.columns)

Index(['BIL', 'BND', 'GLD', 'HYG', 'IEF', 'IWM', 'LQD', 'QQQ', 'SPY', 'TIP',
       'TLT', 'XLB', 'XLE', 'XLF', 'XLI', 'XLK', 'XLP', 'XLRE', 'XLU', 'XLV',
       'XLY'],
      dtype='str', name='Ticker')
Index(['CL=F', 'DX-Y.NYB', 'GC=F', 'HG=F', 'HYG', 'LQD', 'NG=F', 'SHY', 'TLT',
       '^FTSE', '^GSPC', '^HSI', '^MOVE', '^TNX', '^VIX', '^VXN'],
      dtype='str', name='Ticker')


In [25]:
df_merged = pd.merge(prices, prices_indicies, on="Date", how="outer")
df_merged.dropna(how='all',inplace=True)
df_merged.head()

Ticker,BIL,BND,GLD,HYG_x,IEF,IWM,LQD_x,QQQ,SPY,TIP,...,NG=F,SHY,TLT_y,^FTSE,^GSPC,^HSI,^MOVE,^TNX,^VIX,^VXN
Date,,,,,,,,,,,,,,,,,,,,,
1927-12-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,17.660000,NaN,NaN,NaN,NaN,NaN
1928-01-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,17.760000,NaN,NaN,NaN,NaN,NaN
1928-01-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,17.719999,NaN,NaN,NaN,NaN,NaN
1928-01-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,17.549999,NaN,NaN,NaN,NaN,NaN
1928-01-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,17.660000,NaN,NaN,NaN,NaN,NaN


In [26]:
df_merged.to_csv('yfinance_data.csv', index_label='date')